In [18]:
import numpy as np
import matplotlib.pyplot as plt


In [19]:
import os
import urllib.request
import zipfile

classes_to_download = ["walking","boxing", "handclapping"]


os.makedirs("./kth_videos/", exist_ok=True)

print("Starting downloads...")

for action in classes_to_download:
    print(f"\n--- Processing '{action}' ---")
    zip_filename = f"{action}.zip"
    url = f"http://www.csc.kth.se/cvap/actions/{zip_filename}"


    os.system(f"wget -q {url}")


    print(f"Extracting {zip_filename}...")
    os.system(f"unzip -q {zip_filename} -d ./kth_videos/")


    os.system(f"rm {zip_filename}")

print("\nAll selected classes have been downloaded and extracted into ./kth_videos/!")

Starting downloads...

--- Processing 'walking' ---
Extracting walking.zip...

--- Processing 'boxing' ---
Extracting boxing.zip...

--- Processing 'handclapping' ---
Extracting handclapping.zip...

All selected classes have been downloaded and extracted into ./kth_videos/!


In [20]:
import cv2
import numpy as np
import pandas as pd
import os

def process_kth_dataset(video_dir=r"C:\Users\subhe\OneDrive\Desktop\VL-JEPA\codes\kth_videos", output_csv="kth_cctv_features.csv"):
    print(f"Scanning directory {video_dir} for KTH videos...")

    if not os.path.exists(video_dir):
        print(f"Error: Directory {video_dir} not found. Please ensure your videos are extracted here.")
        return 0

    video_files = [f for f in os.listdir(video_dir) if f.endswith('.avi')]
    print(f"Found {len(video_files)} video files. Starting feature extraction...")

    dataset_records = []
    label_to_id = {}
    current_class_id = 0

    for filename in video_files:

        action_label = filename.split('_')[1]

        if action_label not in label_to_id:
            label_to_id[action_label] = current_class_id
            current_class_id += 1

        target_class_id = label_to_id[action_label]
        video_path = os.path.join(video_dir, filename)
        cap = cv2.VideoCapture(video_path)

        # Sample 5 frames per video
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames < 20:
            cap.release()
            continue

        frames_to_sample = np.linspace(10, total_frames - 10, 5, dtype=int)

        for frame_idx in frames_to_sample:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret: continue

            # Apply Simulated CCTV Occlusion Mask
            mask_x1, mask_x2 = 60, 100
            mask_y1, mask_y2 = 40, 80
            cv2.rectangle(frame, (mask_x1, mask_y1), (mask_x2, mask_y2), (0, 0, 0), -1)

            # Convert to HSV and extract 256-bin histogram
            hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
            hist = cv2.calcHist([hsv_frame], [0, 1], None, [16, 16], [0, 180, 0, 256])
            cv2.normalize(hist, hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)

            features = hist.flatten().tolist()

            dataset_records.append({
                "video_file": filename,
                "target_class": target_class_id,
                "hsv_features": features
            })

        cap.release()

    # Save to CSV
    df = pd.DataFrame(dataset_records)
    df.to_csv(output_csv, index=False)

    print("\n--- Extraction Complete! ---")
    print(f"Total frames processed: {len(df)}")
    print(f"Classes found: {label_to_id}")
    print(f"Saved dataset to {output_csv}")

    return len(label_to_id)

num_found_classes = process_kth_dataset()

Scanning directory C:\Users\subhe\OneDrive\Desktop\VL-JEPA\codes\kth_videos for KTH videos...
Found 0 video files. Starting feature extraction...

--- Extraction Complete! ---
Total frames processed: 0
Classes found: {}
Saved dataset to kth_cctv_features.csv


In [21]:
import numpy as np
import pandas as pd
import ast
import matplotlib.pyplot as plt

class FinalNumPyJEPA:
    def __init__(self, input_dim, target_classes, embed_dim=64, lr=0.01):
        self.lr = lr

        self.W_v = np.random.randn(embed_dim, input_dim) * np.sqrt(2. / input_dim)
        self.b_v = np.zeros((embed_dim, 1))

        self.W_y = np.random.randn(embed_dim, target_classes) * np.sqrt(2. / target_classes)
        self.b_y = np.zeros((embed_dim, 1))

        self.W_p = np.random.randn(embed_dim, embed_dim) * np.sqrt(2. / embed_dim)
        self.b_p = np.zeros((embed_dim, 1))

    def relu(self, Z): return np.maximum(0, Z)
    def relu_deriv(self, Z): return Z > 0

    def forward(self, X_v, Y):
        # Encoders
        Z_v = np.dot(self.W_v, X_v) + self.b_v
        self.S_v = self.relu(Z_v)
        self.S_y = np.dot(self.W_y, Y) + self.b_y

        # Predictor
        self.S_y_hat = np.dot(self.W_p, self.S_v) + self.b_p

        # JEPA Embedding Loss
        loss = 0.5 * np.sum((self.S_y_hat - self.S_y)**2)
        return loss, Z_v

    def backward(self, X_v, Y, Z_v):
        d_S_y_hat = self.S_y_hat - self.S_y

        # Predictor Gradients
        d_W_p = np.dot(d_S_y_hat, self.S_v.T)
        d_b_p = d_S_y_hat

        # Visual Encoder Gradients
        d_S_v = np.dot(self.W_p.T, d_S_y_hat)
        d_Z_v = d_S_v * self.relu_deriv(Z_v)
        d_W_v = np.dot(d_Z_v, X_v.T)
        d_b_v = d_Z_v

        # Target Encoder Gradients
        d_S_y = -d_S_y_hat
        d_W_y = np.dot(d_S_y, Y.T)
        d_b_y = d_S_y

        # Weight Updates
        self.W_p -= self.lr * d_W_p; self.b_p -= self.lr * d_b_p
        self.W_v -= self.lr * d_W_v; self.b_v -= self.lr * d_b_v
        self.W_y -= self.lr * d_W_y; self.b_y -= self.lr * d_b_y

def run_training_and_plot(csv_path="kth_cctv_features.csv", num_classes=3, epochs=40):
    print(f"Loading dataset from {csv_path}...")
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print("Error: CSV not found. Did you run the extraction cell first?")
        return

    df['hsv_features'] = df['hsv_features'].apply(ast.literal_eval)


    model = FinalNumPyJEPA(input_dim=256, target_classes=num_classes, embed_dim=64, lr=0.005)
    loss_history = []

    print("Starting Training Simulation...")
    for epoch in range(epochs):
        df_shuffled = df.sample(frac=1).reset_index(drop=True)
        epoch_loss = 0

        for index, row in df_shuffled.iterrows():
            X_v = np.array(row['hsv_features']).reshape(256, 1)
            Y = np.zeros((num_classes, 1))
            Y[int(row['target_class']), 0] = 1.0

            loss, Z_v = model.forward(X_v, Y)
            model.backward(X_v, Y, Z_v)
            epoch_loss += loss

        avg_loss = epoch_loss / len(df)
        loss_history.append(avg_loss)
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | Average JEPA Latent Loss: {avg_loss:.4f}")


    plt.figure(figsize=(9, 5))
    plt.plot(loss_history, marker='o', linestyle='-', color='#1f77b4', linewidth=2)
    plt.title('VL-JEPA: NumPy Embedding-Space Training Loss', fontsize=14, fontweight='bold')
    plt.xlabel('Training Epochs', fontsize=12)
    plt.ylabel('L2 Latent Distance', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()


try:
    run_training_and_plot(num_classes=num_found_classes, epochs=40)
except NameError:

    print("Please manually enter the number of classes found:")
    manual_classes = int(input("> "))
    run_training_and_plot(num_classes=manual_classes, epochs=40)

Loading dataset from kth_cctv_features.csv...


EmptyDataError: No columns to parse from file